# 04 — Frontier-Vergleich (Phase 5)

Eure Hand-Annotation aus Phase 2 (`annotation/meine_gold.csv`) gegen Frontier-LLM-Annotation derselben 12 Anzeigen (`annotation/frontier_gold.csv`). Output: κ-Tabelle, drei Disagreement-Beispiele, Material fürs Make-or-Buy-Memo (`memo_make_or_buy.md` im Repo-Root).

Cheatsheet: `CHEATSHEETS/frontier-llm-workflow.md`.

## Run-Header

| Feld | Wert |
|---|---|
| Frontier-Modell | GPT-5.5 (in ChatGPT)|
| Datum | 2026-05-18 |
| Prompt-Variante | v1 — Schema + Negativ-Definitionen + 3 Few-Shots aus meine_gold (Peagle, Wilken, Hays) |
| Anzahl Korrektur-Turns | 0 (Frontier hat beim ersten Versuch schema-konform geantwortet) |
| Auffälligkeiten | keine — alle 9 zurückgegebenen Anzeigen schema-konform |
| Predictions-Quelle | `annotation/frontier_gold.csv` (9 ChatGPT + 3 Few-Shot-Übernahmen aus `meine_gold.csv`) |
| Vergleichs-Basis | `annotation/meine_gold.csv` (Hand-Annotation aus Phase 2) |

**Methodischer Hinweis:** 3 der 12 Anzeigen wurden ChatGPT als Few-Shot-Beispiele gegeben (Peagle, Wilken, Hays). Diese 3 werden im κ-Compute **ausgeschlossen**, weil ChatGPT die "richtigen" Werte ja schon im Prompt gesehen hat. Effektive Stichprobengröße für κ: **n=9**.

**Phase-2-Hinweis:** Die zweite Hand-Annotation (`partner_gold.csv`) wurde mit Claude erstellt. Phase 5 nutzt jetzt ChatGPT (anderes Modell, andere Provider-Familie), um echten Frontier-Mensch-Vergleich zu haben.

## Frontier-Daten laden + Schema-Konformität prüfen

In [1]:
"""
ChatGPT-Output -> frontier_gold.csv
Inkl. die 3 Few-Shot-Anzeigen aus meine_gold.csv übernehmen, damit die CSV
am Ende alle 12 IDs hat (sonst hätte κ nur 9 gemeinsame).

WICHTIG für die κ-Diskussion: die 3 Few-Shot-Anzeigen werden BEIM κ-COMPUTE
ausgeschlossen (sonst künstlich perfekte Übereinstimmung). Wir markieren sie
mit einer Notiz.
"""
import json
import csv
from pathlib import Path

# ChatGPT-Antwort als String einfügen (hier direkt eingebettet)
chatgpt_antwort = """[
{"id": "13644-294294-S","homeoffice": "nicht_genannt","vertragsart": "festanstellung","erfahrungslevel": "nicht_genannt","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": ["python", "sql", "azure"]},
{"id": "10000-1185445644-S","homeoffice": "ja","vertragsart": "festanstellung","erfahrungslevel": "mid","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": ["python", "java", "spark"]},
{"id": "10001-1002026714-S","homeoffice": "nicht_genannt","vertragsart": "ausbildung","erfahrungslevel": "junior","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": []},
{"id": "15719-44156922-55-S","homeoffice": "teilweise","vertragsart": "festanstellung","erfahrungslevel": "nicht_genannt","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": ["python", "sql", "git"]},
{"id": "19384-938440348-S","homeoffice": "teilweise","vertragsart": "festanstellung","erfahrungslevel": "junior","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": ["r", "sql", "gitlab"]},
{"id": "10001-1003030981-S","homeoffice": "nicht_genannt","vertragsart": "ausbildung","erfahrungslevel": "junior","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": []},
{"id": "16818-100814418-S","homeoffice": "nicht_genannt","vertragsart": "ausbildung","erfahrungslevel": "junior","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": ["ms-office", "java"]},
{"id": "13644-297872-S","homeoffice": "nicht_genannt","vertragsart": "festanstellung","erfahrungslevel": "mid","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": ["python", "pandas", "scikit-learn"]},
{"id": "10001-1002616378-S","homeoffice": "nicht_genannt","vertragsart": "festanstellung","erfahrungslevel": "mid","gehalt_min_eur": null,"gehalt_zeitraum": null,"skills_top3": ["python", "azure", "scikit-learn"]}
]"""

annotationen_frontier = json.loads(chatgpt_antwort)
print(f"ChatGPT-Antworten: {len(annotationen_frontier)} Anzeigen")

# Few-Shot-IDs aus meine_gold übernehmen (markiert als 'few_shot' in notiz)
FEW_SHOT_REFNRS = {
    "10001-1002678126-S",         # Peagle (Festanstellung, junior)
    "16818-100787927-S",          # Wilken (Ausbildung mit 953€)
    "13319-868490/1_607417LS-S",  # Hays (Senior, ja-homeoffice)
}

# meine_gold.csv laden, um Few-Shot-Werte zu übernehmen
meine_gold = {}
with open("../annotation/meine_gold.csv", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        if row["id"].strip():
            meine_gold[row["id"]] = row

# CSV-Spalten
spalten = ["id", "homeoffice", "vertragsart", "erfahrungslevel",
           "gehalt_min_eur", "gehalt_zeitraum", "skills_top3", "notiz"]

pfad = Path("../annotation/frontier_gold.csv")
with pfad.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=spalten)
    writer.writeheader()
    
    # 1. Die 9 ChatGPT-Annotationen schreiben
    for a in annotationen_frontier:
        skills = a.get("skills_top3", [])
        skills_str = "|".join(skills) if skills else ""
        
        gehalt = a.get("gehalt_min_eur")
        gehalt_str = str(gehalt) if gehalt is not None else ""
        
        zeitraum = a.get("gehalt_zeitraum")
        zeitraum_str = zeitraum if zeitraum else ""
        
        writer.writerow({
            "id": a["id"],
            "homeoffice": a["homeoffice"],
            "vertragsart": a["vertragsart"],
            "erfahrungslevel": a["erfahrungslevel"],
            "gehalt_min_eur": gehalt_str,
            "gehalt_zeitraum": zeitraum_str,
            "skills_top3": skills_str,
            "notiz": "ChatGPT GPT-X annotiert",
        })
    
    # 2. Die 3 Few-Shot-Anzeigen aus meine_gold übernehmen (zur Vollständigkeit)
    for refnr in FEW_SHOT_REFNRS:
        if refnr in meine_gold:
            row = meine_gold[refnr].copy()
            row["notiz"] = "FEW_SHOT — aus meine_gold übernommen, NICHT in κ einrechnen"
            writer.writerow({k: row.get(k, "") for k in spalten})

print(f"✓ {pfad} geschrieben mit 12 Einträgen (9 ChatGPT + 3 Few-Shot-Übernahmen)")
print(f"  Die 3 Few-Shot-IDs werden im κ-Compute später ausgeschlossen.")

ChatGPT-Antworten: 9 Anzeigen
✓ ../annotation/frontier_gold.csv geschrieben mit 12 Einträgen (9 ChatGPT + 3 Few-Shot-Übernahmen)
  Die 3 Few-Shot-IDs werden im κ-Compute später ausgeschlossen.


## κ Mensch ↔ Frontier

Bevor ich die κ-Werte berechne, formuliere ich folgende Erwartungen, basierend auf:
(a) den Phase-2-κ-Werten Mensch↔Claude (1.0/1.0/0.625),  
(b) den Phase-3-Befunden (welche Felder ChatGPT auch ohne Few-Shots gut hinbekommt),  
(c) der Beobachtung, dass ChatGPT meine 3 Few-Shots eingebaut hat (was den κ leicht künstlich erhöhen kann auch außerhalb der Few-Shots, durch Stil-Transfer).

### Erwartung pro Feld

| Feld | Erwartetes κ | Begründung |
|---|---|---|
| `vertragsart` | **fast perfekt (≥0.85)** | Ausbildung vs. Festanstellung textuell sehr eindeutig — sowohl Mensch als auch jedes halbwegs gute LLM trifft das fast immer. |
| `homeoffice` | **substanziell (0.6-0.8)** | Hauptrisiko: "Home Office möglich" interpretiert ChatGPT vermutlich anders als ich (z.B. als `ja` statt `teilweise`). |
| `erfahrungslevel` | **moderat (0.4-0.6)** — schwächstes Feld | Schon Mensch↔Claude hatte 0.625. Das Schema hat hier eindeutige Grauzonen (mid vs. nicht_genannt bei "Erfahrung erwähnt aber ohne Jahresangabe"). |

**Gesamt-κ-Erwartung:** mittlere bis substanzielle Übereinstimmung, getragen durch `vertragsart`. Schwachpunkt bleibt `erfahrungslevel`.

### Erwartung für Make-or-Buy

Wenn `vertragsart` und `homeoffice` ≥0.7 → Frontier ist auf diesen Feldern produktionstauglich.  
Wenn `erfahrungslevel` <0.5 → Frontier nur unter Validation/Review für dieses Feld einsetzbar.  
Möglicher Hybrid-Ansatz: einfache Felder ans Frontier, kritische Felder Mensch oder Mensch+Validator.

In [2]:
"""
Cohen's kappa Mensch ↔ Frontier.
Wichtig: Few-Shot-Anzeigen ausschließen, sonst künstlich perfekte Übereinstimmung.
"""
import csv
import sys
sys.path.append("../annotation")
from validate import cohen_kappa, load_annotations
from pathlib import Path

FEW_SHOT_REFNRS = {
    "10001-1002678126-S",
    "16818-100787927-S",
    "13319-868490/1_607417LS-S",
}

# Annotationen laden
mensch = load_annotations(Path("../annotation/meine_gold.csv"))
frontier = load_annotations(Path("../annotation/frontier_gold.csv"))

# Few-Shots ausschließen
gemeinsame_ids = sorted(
    (set(mensch) & set(frontier)) - FEW_SHOT_REFNRS,
    key=lambda x: int(x.split("-")[0])
)
print(f"Gemeinsame IDs nach Few-Shot-Ausschluss: {len(gemeinsame_ids)}\n")
print(f"Few-Shots ausgeschlossen (im Prompt verraten): {sorted(FEW_SHOT_REFNRS)}\n")

# κ pro Feld
KATEGORIAL = ["homeoffice", "vertragsart", "erfahrungslevel"]

print(f"{'Feld':<20} {'κ':>8} {'Übereinst.':>12} {'Interpretation':<25}")
print("─" * 70)

def interpret(k):
    if k > 0.80: return "fast perfekt"
    if k > 0.60: return "substanziell"
    if k > 0.40: return "moderat"
    if k > 0.20: return "mäßig"
    if k > 0.00: return "schlecht"
    return "schlechter als Zufall"

for feld in KATEGORIAL:
    labels_m = [(mensch[i].get(feld) or "").strip() for i in gemeinsame_ids]
    labels_f = [(frontier[i].get(feld) or "").strip() for i in gemeinsame_ids]
    k = cohen_kappa(labels_m, labels_f)
    agree = sum(1 for a, b in zip(labels_m, labels_f) if a == b)
    pct = agree / len(gemeinsame_ids)
    print(f"{feld:<20} {k:>8.3f} {pct:>11.0%}  {interpret(k):<25}")

Gemeinsame IDs nach Few-Shot-Ausschluss: 9

Few-Shots ausgeschlossen (im Prompt verraten): ['10001-1002678126-S', '13319-868490/1_607417LS-S', '16818-100787927-S']

Feld                        κ   Übereinst. Interpretation           
──────────────────────────────────────────────────────────────────────
homeoffice              0.769         89%  substanziell             
vertragsart             1.000        100%  fast perfekt             
erfahrungslevel         0.836         89%  fast perfekt             


## Drei Disagreement-Beispiele

Pro Beispiel: Anzeige-ID, euer Wert, Frontier-Wert, **wer hatte recht** — Begründung mit Bezug auf Schema-Definition oder konkrete Anzeigen-Stelle. Material fürs Make-or-Buy-Memo.

In [3]:
"""
Welche Anzeigen sind die Disagreements? Pflicht laut Aufgabenblatt: 3 konkrete Fälle.
"""
print("="*80)
print("DISAGREEMENTS Mensch ↔ Frontier (ChatGPT)")
print("="*80)

for feld in KATEGORIAL + ["gehalt_min_eur", "gehalt_zeitraum", "skills_top3"]:
    fehler = []
    for refnr in gemeinsame_ids:
        m_val = (mensch[refnr].get(feld) or "").strip()
        f_val = (frontier[refnr].get(feld) or "").strip()
        if m_val != f_val:
            fehler.append((refnr, m_val, f_val))
    
    if fehler:
        print(f"\n{feld} ({len(fehler)} Disagreement(s)):")
        for refnr, m_val, f_val in fehler:
            print(f"  {refnr}")
            print(f"     Mensch:   {m_val!r}")
            print(f"     Frontier: {f_val!r}")
    else:
        print(f"\n{feld}: ✓ vollständige Übereinstimmung")

DISAGREEMENTS Mensch ↔ Frontier (ChatGPT)

homeoffice (1 Disagreement(s)):
  10000-1185445644-S
     Mensch:   'teilweise'
     Frontier: 'ja'

vertragsart: ✓ vollständige Übereinstimmung

erfahrungslevel (1 Disagreement(s)):
  10000-1185445644-S
     Mensch:   'senior'
     Frontier: 'mid'

gehalt_min_eur: ✓ vollständige Übereinstimmung

gehalt_zeitraum: ✓ vollständige Übereinstimmung

skills_top3 (7 Disagreement(s)):
  10000-1185445644-S
     Mensch:   'python|spark|ci/cd'
     Frontier: 'python|java|spark'
  10001-1002616378-S
     Mensch:   'python|machine learning|azure'
     Frontier: 'python|azure|scikit-learn'
  13644-294294-S
     Mensch:   'python|sql|r'
     Frontier: 'python|sql|azure'
  13644-297872-S
     Mensch:   'python|machine learning|pandas'
     Frontier: 'python|pandas|scikit-learn'
  15719-44156922-55-S
     Mensch:   'python|machine learning|SQL'
     Frontier: 'python|sql|git'
  16818-100814418-S
     Mensch:   'java|ms office'
     Frontier: 'ms-office|java'
 